# Project 1
In this project, I investigated one year sales from an online shop. The shop sells gifts and homeware. In this data set, each row represents one product line in one order. The data is from Kaggle with this link: 
[An Online Shop Business (Kaggle)](https://www.kaggle.com/datasets/gabrielramos87/an-online-shop-business)

I will use the *`Price`* column to answer a simple question: "What is the cost of a typical item in this shop?"

## Data Cleaning

In [14]:
import pandas as pd

# Load the sales transactions
sales_raw = pd.read_csv("sales_transaction.csv")

# Show the first few rows
sales_raw.head()

,TransactionNo,Date,ProductNo,ProductName,Price,Quantity,CustomerNo,Country
0,581482,12/9/2019,22485,Set Of 2 Wooden Market Crates,21.47,12,17490.0,United Kingdom
1,581475,12/9/2019,22596,Christmas Star Wish List Chalkboard,10.65,36,13069.0,United Kingdom
2,581475,12/9/2019,23235,Storage Tin Vintage Leaf,11.53,12,13069.0,United Kingdom
3,581475,12/9/2019,23272,Tree T-Light Holder Willie Winkie,10.65,12,13069.0,United Kingdom
4,581475,12/9/2019,23239,Set Of 4 Knick Knack Tins Poppies,11.94,6,13069.0,United Kingdom


Each row tells us the transaction number, customer number, date, price, product name, quantity and so on. Let's check out how big the data set is.

In [15]:
# Number of rows and columns, plus the type of each column
sales_raw.info()


<class 'pandas.DataFrame'>
RangeIndex: 536350 entries, 0 to 536349
Data columns (total 8 columns):
 #   Column         Non-Null Count   Dtype  
---  ------         --------------   -----  
 0   TransactionNo  536350 non-null  str    
 1   Date           536350 non-null  str    
 2   ProductNo      536350 non-null  str    
 3   ProductName    536350 non-null  str    
 4   Price          536350 non-null  float64
 5   Quantity       536350 non-null  int64  
 6   CustomerNo     536295 non-null  float64
 7   Country        536350 non-null  str    
dtypes: float64(2), int64(1), str(5)
memory usage: 32.7 MB


The data set is quite large with over half a million rows and 8 columns. `Price` and `Quantity` are stored as numbers while `TransactionNo` is stored as text which indicates that some transaction numbers contain letters. What is the time range of this data?

In [16]:
# Convert the Date column from text into real dates
sale_dates = pd.to_datetime(sales_raw["Date"], format="%m/%d/%Y")

print(f"First sale: {sale_dates.min():%B %d, %Y}")
print(f"Last sale:  {sale_dates.max():%B %d, %Y}")


First sale: December 01, 2018
Last sale:  December 09, 2019


Now, let's get a quick summary of two numeric columns to see if there are something unusual. 

In [17]:
# Summary statistics for the numeric columns
sales_raw[["Price", "Quantity"]].describe()


,Price,Quantity
count,536350.000000,536350.000000
mean,12.662182,9.919347
std,8.490450,216.662300
min,5.130000,-80995.000000
25%,10.990000,1.000000
50%,11.940000,3.000000
75%,14.090000,10.000000
max,660.620000,80995.000000


The minimum `Quantity` is negative. We cannot buy negative amounts of goods. Let's find out which rows those are.

In [18]:
# Rows with cancelled transactions (transaction number starting with "C")
is_cancelled = sales_raw["TransactionNo"].astype(str).str.startswith("C")

# Rows with a negative quantity
is_negative = sales_raw["Quantity"] < 0

# Rows that are both cancelled and have a negative quantity
is_both = is_cancelled & is_negative

print(f"Cancelled rows:          {is_cancelled.sum():,}")
print(f"Negative-quantity rows:  {is_negative.sum():,}")
print(f"Rows that are both:      {is_both.sum():,}")

Cancelled rows:          8,585
Negative-quantity rows:  8,585
Rows that are both:      8,585


Negative quantities come from **cancelled orders**. They are transactions whose transaction number starts with "C". I will remove the cancelled transactions and analyze only completed sales. 

In [19]:
# Keep only completed (non-cancelled) sales
sales = sales_raw[~is_cancelled]
print(f"Rows kept for analysis: {len(sales):,}")


Rows kept for analysis: 527,765


Now, the data is clean. We can now go to calculate the mean, median and mode of `Price`

## Mean, Median and Mode of Prices

In [20]:
# Only the Price column is used
prices = sales["Price"]

price_mean = prices.mean()
price_median = prices.median()

# mode() returns a Series of values because multiple price points
# could have the same highest frequency. 
price_modes = prices.mode()
mode_count = (prices == price_modes[0]).sum()

print(f"Mean price:   {price_mean:.2f}")
print(f"Median price: {price_median:.2f}")
print(f"Mode price:   {', '.join(f'{m:.2f}' for m in price_modes)}")
print(f"The mode appears {mode_count:,} times")


Mean price:   12.63
Median price: 11.94
Mode price:   11.53
The mode appears 44,813 times


The mean of price is higher than its median. This tells us that the distribution of prices is right-skewed. A few expensive items pull the average up. Most items are cheaper than the mean. Thus, the median price reflects the cost of a typical item better. The mode is slightly below the median. It appears tens of thousands of times. This suggests that many products are sold at the same few price points.

In answer to the question, a typical item costs a bit under £12. 

## Doing It the Hard Way

We can also read the file and calculate statistics without using pandas and the statistic module. In other words, we will not access the DataFrame or Series. If we move the code in a different notebook, it will still work.

In [21]:
import csv

# Read the file again and collect prices of completed transactions. 
# As each value is read as text first, we need to convert prices 
# to numbers.
price_list = []

with open("sales_transaction.csv", newline="", encoding="utf-8") as file:
    reader = csv.DictReader(file)
    for row in reader:
        # Skip if orders are cancelled
        if row["TransactionNo"].startswith("C"):
            continue
        price_list.append(float(row["Price"]))

print(f"Prices collected: {len(price_list):,}")


Prices collected: 527,765


The number of rows we got is the same as what we got using pandas. Now let us compute the mean, median and mode of prices.

In [22]:
# Mean: sum up the prices and divide them by the number of prices
price_total = 0
for price in price_list:
    price_total += price

hard_mean = price_total / len(price_list)
print(f"Mean price:   {hard_mean:.2f}")


Mean price:   12.63


In [23]:
# Median: the middle value of the price list once it is sorted
sorted_prices = sorted(price_list)
price_count = len(sorted_prices)
middle = price_count // 2

if price_count % 2 == 1:
    # Odd count: there is a single middle value
    hard_median = sorted_prices[middle]
else:
    # Even count: average the two middle values
    hard_median = (sorted_prices[middle - 1] + sorted_prices[middle]) / 2

print(f"Median price: {hard_median:.2f}")


Median price: 11.94


In [24]:
# Mode: Use a dictionary to count how many times
# each price appears: {price: count}
# select prices with highest count
price_counts = {}
for price in price_list:
    if price in price_counts:
        price_counts[price] += 1
    else:
        price_counts[price] = 1

# Find the highest count
highest_count = 0
for count in price_counts.values():
    if count > highest_count:
        highest_count = count

# Collect every price with that count, in case of a tie
hard_modes = []
for price, count in price_counts.items():
    if count == highest_count:
        hard_modes.append(price)

mode_text = ", ".join(f"{mode:.2f}" for mode in hard_modes)
print(f"Mode price:   {mode_text}")
print(f"The mode appears {highest_count:,} times")


Mode price:   11.53
The mode appears 44,813 times


In [25]:
# Check that numbers in both pandas and hard sections agree
print("Mean matches:  ", round(hard_mean, 2) == round(price_mean, 2))
print("Median matches:", round(hard_median, 2) == round(price_median, 2))
print("Mode matches:  ", sorted(hard_modes) == sorted(price_modes))


Mean matches:   True
Median matches: True
Mode matches:   True


We made it! The calculated mean, median and mode using the hard way match their counterparts using pandas. In this section, I used for loops, sort and dictionary to do the work that is done by pandas using one line `prices.mean()`, `prices.median()`, or `prices.mode()`. Now we know what is happening underneath. 

## Visualization of Prices
Let's draw a text histogram to see how many items sold in each £1 price range as numbers alone don't show the shape of prices.

Most prices are in the £10–£20 range. Only a few items cost hundreds of pounds. I will group every price from £25 up into one row so that the chart is readable. 

In [26]:
price_cap = 25        # prices at or above this go into one "25+" row
max_bar_width = 50    # length of the longest bar, in characters
bar_symbol = "\u2588"

# Round each price down to a whole number to get its £1 bucket,
# then put everything at or above the cap into a single bucket
price_buckets = (prices // 1).astype(int).clip(upper=price_cap)

# Count how many items fall in each bucket, cheapest first
bucket_counts = price_buckets.value_counts().sort_index()

# Scale the bars so the biggest bucket is max_bar_width symbols long
largest_count = bucket_counts.max()

# Title
print("Items sold by unit price, completed sales")
print()

# One row per price bucket
for bucket, count in bucket_counts.items():
    # Label the row
    if bucket == price_cap:
        label = f"£{bucket}+"
    else:
        label = f"£{bucket}–{bucket + 1}"

    bar_length = round(count / largest_count * max_bar_width)
    bar = bar_symbol * bar_length

    print(f"{label:>7} | {bar} {count:,}")


Items sold by unit price, completed sales

   £5–6 | ████ 12,896
   £6–7 | ████████████████ 48,130
   £7–8 | ██████ 16,995
   £8–9 |  48
  £9–10 |  12
 £10–11 | ███████████████████ 54,731
 £11–12 | ██████████████████████████████████████████████████ 146,620
 £12–13 | ██████████████████████ 64,728
 £13–14 | ███████████████ 44,326
 £14–15 | ███████████████ 42,677
 £15–16 | ████████ 23,047
 £16–17 | ██████ 16,960
 £17–18 | ███ 8,294
 £18–19 | ███████ 21,894
 £19–20 |  1,156
 £20–21 | ██ 6,052
 £21–22 | █ 3,880
 £22–23 |  559
 £23–24 | ██ 6,065
 £24–25 |  567
   £25+ | ███ 8,128


There is a huge peak at £11–12. Both the median and the mode fall in this bucket. It is by far the most common price range. One surprising point is that there is almost nothing sold at £8-10, although there are plenty of items sold on either side. This may relate to the store's pricing decisions. Also from the plot, we can see that there is a long tail of expensive items. They only account for a small share of items but their high prices pull the mean price up. The plot supports our previous answer. A typical item in this shop costs a bit under £12.  

## Conclusion

In conclusion, a typical item in this shop costs a bit under £12. We got this from our median and mode numbers. Both are slightly below £12 while the mean is pulled up by a few expensive items. The text histogram shows a huge peak in the price bucket of £11-12. And prices cluster tightly around £10-15 with an unusual gap at £8-10. 

Note that this conclusion applies only to non-cancelled orders in this shop over one year (Dec 2018 - Dec 2019) and the price is *price per unit* rather than total value of an order. 